In [18]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [19]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))
docs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "docs_by_item.parquet"))


In [20]:
df['outcome'] = df['project_result'].map({
    'APPROVED': 'Approved',
    'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'DENIED': 'Delayed / Denied',
    'APPLICATION WITHDRAWN': 'Delayed / Denied',
    'DELAYED': 'Delayed / Denied'
})

In [21]:
n_approved = f"{(df['outcome'] == "Approved").sum():,.0f}"
n_approved_w_conditions = f"{(df['outcome'] == "Approved w/ conditions").sum():,.0f}"
n_delayed_denied = f"{(df['outcome'] == "Delayed / Denied").sum():,.0f}"

In [22]:
header = fr"""
\begin{{table}}[H]
\caption{{Descriptive Statistics by Motion Outcome}}
\vspace{{0.2cm}}
\label{{tab_bivariate_descriptives}}
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{threeparttable}}
\begin{{tabular}}{{lrrrrr}} \toprule
 Variable & \makecell{{Approved \\ ($n = {n_approved}$)}} & \makecell{{Approved w/ \\ conditions \\ ($n = {n_approved_w_conditions}$)}} & \makecell{{Delayed / \\ denied \\ ($n = {n_delayed_denied}$)}} & \makecell{{Test \\ stat}} & p-value  \\ \midrule
"""
footer = r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This reports descriptive statistics for variables grouped by motion outcome. To test differences between groups, a one-way ANOVA F-test was conducted for continuous variables and a chi-square test was conducted for categorical variables.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""


In [23]:
with open(os.path.join(LOCAL_PATH, "tables", "tab_bivariate_descriptives.tex"), "w", encoding='utf-8') as f:
    f.write(header + footer)
